In [ ]:
from dataclasses import dataclass
from pathlib import Path
import pandas as pd


### Data Import & Path Setup

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PTConfig:
    state: str
    payer: str
    file: str
    output_name: str | None = None

cfg = PTConfig(
    state="Florida",
    payer="BUCA",
    file="CA BUCA 2026-03-28",
    output_name=None,   # optional
)

In [34]:
# =========================
# Permanent Roots
# =========================
ROOT = Path(r"C:\Users\pirat\Dropbox\Consulting Inc")
AD_ROOT = ROOT / "AAD" / "Derm Group Analysis"
# AD_ROOT = ROOT / "Advanced Dermatology" / "1. Payor Data & Contracts"
USMDB_IMPORT = ROOT / "USMDB" / "Data_Import"
USMDB_EXPORT = ROOT / "USMDB" / "Data_Export"
CMS_ROOT = ROOT / "Josh Perkins Data" / "CMS Data"

SERVICE_MAP_PATH = CMS_ROOT / "Service Category Mapping.xlsx"
EIN_CROSSWALK_PATH = USMDB_IMPORT / "Org_EIN_NPI_Crosswalk_2026_02_PriceMedic.csv"
GEOGRAPHY_PATH = USMDB_EXPORT / "00a_zip.csv"
REGION_PATH = USMDB_IMPORT / "Regions.xlsx"
DERMATOLOGY_ORGS_PATH = USMDB_EXPORT / "05_dermatology_clean.csv"

# =========================
# Path Helpers
# =========================
def get_pt_dir(cfg: PTConfig) -> Path:
    return AD_ROOT / cfg.state / f"{cfg.payer} Analysis" / "Tableau"


def get_input_paths(cfg: PTConfig) -> dict[str, Path]:
    pt_dir = get_pt_dir(cfg)
    return {
        "pt_raw": pt_dir / f"{cfg.file}.csv",
        "service_map": SERVICE_MAP_PATH,
        "ein_crosswalk": EIN_CROSSWALK_PATH,
        "geography": GEOGRAPHY_PATH,
        "regions": REGION_PATH,
        "derm_orgs": DERMATOLOGY_ORGS_PATH,
    }


def validate_paths(paths: dict[str, Path]) -> None:
    missing = [f"{k}: {v}" for k, v in paths.items() if not v.exists()]
    if missing:
        raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing))


# =========================
# Load Inputs
# =========================
def load_inputs(cfg: PTConfig) -> dict[str, pd.DataFrame]:
    paths = get_input_paths(cfg)
    validate_paths(paths)
    return {
        "pt_raw": pd.read_csv(
            paths["pt_raw"],
            dtype={
                "billing_code": "string",
                "billing_code_modifier": "string",
                "zip_code": "string",
                "locality": "string",
                "tin": "string",
                "org_ein": "string",
                "cbsa_code": "string",
                "org_est_count_md": "Int64",
                "org_est_count_app": "Int64",
                "org_est_count_npi1": "Int64",
                "cbsa_metro_micro_indicator": "Int64",
            },
            parse_dates=["date", "expiration_date"],
        ),
        "service_map": pd.read_excel(paths["service_map"]),
        "ein_crosswalk": (
            pd.read_csv(
                paths["ein_crosswalk"],
                dtype={
                    "ein": "string",
                    "npi": "string",
                },
            )
            .sort_values(["ein", "npi"])
            .reset_index(drop=True)
        ),
        "geography": pd.read_csv(paths["geography"]),
        "regions": pd.read_excel(paths["regions"]),
        "derm_orgs": pd.read_csv(paths["derm_orgs"]),
    }

# =========================
# Export Helpers
# =========================
def get_export_paths(cfg: PTConfig) -> dict[str, Path]:
    pt_dir = get_pt_dir(cfg)
    base_name = cfg.output_name or f"{cfg.payer}_{cfg.state}_Price_Transparency_Cleaned"
    base = pt_dir / base_name
    return {
        "base": base,
        "xlsx": base.with_suffix(".xlsx"),
        "hyper": base.with_suffix(".hyper"),
    }

In [35]:
inputs = load_inputs(cfg)

pt_raw = inputs["pt_raw"]
service_map = inputs["service_map"]
ein_crosswalk = inputs["ein_crosswalk"]
geography = inputs["geography"]
regions = inputs["regions"]
derm_orgs = inputs["derm_orgs"]

### Data Cleaning

In [36]:
pt_clean = pt_raw.copy()

# if org_name is missing, fill with entity_name
pt_clean["org_name"] = pt_clean["org_name"].fillna(pt_clean["entity_name"])

In [37]:
# convert geography ZIP_Code to string with leading zeros
geography["ZIP_Code"] = geography["ZIP_Code"].astype(str).str.zfill(5)

# merge geography County_ST onto pt_clean
pt_clean = pt_clean.merge(geography[["ZIP_Code", "County_Short", "County_ST"]],
                          how="left", left_on="zip_code", right_on="ZIP_Code").drop(columns=["ZIP_Code"])

In [38]:
cols = ["cbsa_name", "state", "zip_code", "County_Short", "County_ST"]

pt_clean = pt_clean.assign(_row_order=range(len(pt_clean)))

pt_clean = pt_clean.sort_values(
    ["org_name", "org_est_count_npi1"],
    ascending=[True, False],
    na_position="last"
)

pt_clean[cols] = pt_clean[cols].fillna(
    pt_clean.groupby("org_name")[cols].transform("first")
)

pt_clean = (
    pt_clean
    .sort_values("_row_order")
    .drop(columns="_row_order")
    .reset_index(drop=True)
)

In [39]:
# build one-row-per-EIN lookup from ein_crosswalk
npi_lookup = (
    ein_crosswalk[["ein", "npi"]]
    .dropna(subset=["ein", "npi"])
    .drop_duplicates()
    .sort_values(["ein", "npi"])
    .groupby("ein", as_index=False)
    .agg(org_npi_list=("npi", list))
)
# split list into first 3 NPIs + all NPIs if more than 3
npi_lookup["org_npi_1"] = npi_lookup["org_npi_list"].str[0]
npi_lookup["org_npi_2"] = npi_lookup["org_npi_list"].str[1]
npi_lookup["org_npi_3"] = npi_lookup["org_npi_list"].str[2]
npi_lookup["org_npi_all"] = npi_lookup["org_npi_list"].str.join(",")

# keep only the columns you want to merge in & merge into pt_clean using org_ein -> ein
npi_lookup = npi_lookup[["ein", "org_npi_1", "org_npi_2", "org_npi_3", "org_npi_all"]]
 
pt_clean = pt_clean.merge(
    npi_lookup[["ein", "org_npi_1", "org_npi_2", "org_npi_3", "org_npi_all"]],
    how="left",
    left_on="org_ein",
    right_on="ein",
).drop(columns="ein")

In [40]:
# merge "Category Name" from service_map onto pt_clean
pt_clean = pt_clean.merge(
    service_map[["Code Core", "Category Name"]],
    how="left",
    left_on="billing_code",
    right_on="Code Core"
).drop(columns="Code Core").rename(columns={"Category Name": "billing_code_category"})

In [41]:
# merge region from regions onto pt_clean based on County, ST
pt_clean = pt_clean.merge(
    regions[["County, ST", "Region"]],
    how="left",
    left_on="County_ST",
    right_on="County, ST"
).drop(columns="County, ST").rename(columns={"Region": "region"})

In [42]:

# remove all rows where billing_code_modifier is not null
pt_clean = pt_clean.loc[
    pt_clean["billing_code_modifier"].isna()
    | (pt_clean["billing_code_modifier"].astype(str).str.strip() == "00")
].reset_index(drop=True)
# de-dupe rows and reset index
pt_clean = pt_clean.drop_duplicates().reset_index(drop=True)


In [43]:
# format for Tableau export: convert all-null columns to string (Tableau doesn't like nulls in numeric columns)
all_null_cols = [c for c in pt_clean.columns if pt_clean[c].isna().all()]
print(all_null_cols)

for c in all_null_cols:
    pt_clean[c] = pt_clean[c].astype("string")

['billing_code_modifier', 'additional_information']


### Derm Orgs Merge

### Output

In [44]:
import pantab as pt

exports = get_export_paths(cfg)

pt.frame_to_hyper(
    pt_clean,
    exports["hyper"],
    table="Extract",
    table_mode="w",
    atomic=True,
)

# pt_clean.to_excel(exports["xlsx"], index=False)

print(exports["hyper"])
print(exports["xlsx"])

C:\Users\pirat\Dropbox\Consulting Inc\AAD\Derm Group Analysis\Florida\BUCA Analysis\Tableau\BUCA_Florida_Price_Transparency_Cleaned.hyper
C:\Users\pirat\Dropbox\Consulting Inc\AAD\Derm Group Analysis\Florida\BUCA Analysis\Tableau\BUCA_Florida_Price_Transparency_Cleaned.xlsx
